# NYC Yellow Taxi — EDA Scratchpad

Early exploration: schema inspection, dirty-data audit, and sample-design
discovery. The polished analysis and management summary are in `analysis.ipynb`.

See `README.md` for environment setup and reproduction instructions.

## 1. Setup & Ingestion

Single in-memory source of truth: the CSV is read **once** with Polars, then
queried through **DuckDB** by referencing the frame directly (zero-copy). This
gives us a clean dataframe API *and* SQL over the same object, with no duplication.

In [ ]:
import polars as pl
import duckdb

from pathlib import Path
_data = Path('data/Yellow_Taxi_Assignment.csv')
if not _data.exists():
    _data = Path('../data/Yellow_Taxi_Assignment.csv')
CSV_PATH = _data

# infer_schema_length raised well above the default 100: the fee columns are
# empty in early years and would be mis-typed from a short sample.
# try_parse_dates lets Polars type the pickup/dropoff columns as Datetime directly.
trips = pl.read_csv(CSV_PATH, infer_schema_length=100_000, try_parse_dates=True)

# duckdb resolves `trips` from local scope on every query below — no registration needed.
trips.shape

## 2. Schema & Types

First read of the contract: what each column *is*. Two early smells to confirm
later — `airport_fee` arrives as text (it's empty for most years, so no numeric
type was inferred), and `passenger_count` / `RatecodeID` are floats only because
nulls forced them off integer.

In [ ]:
duckdb.sql("describe trips").pl()

## 3. General Numbers (headline KPIs — *raw, pre-cleaning*)

A first sense of scale. These are computed on the **raw** data and therefore
include dirty rows (negative fares, a 177k-mile trip, etc.), so treat them as
upper-bound sanity checks, not final figures. We'll recompute on the cleaned
base once the cleaning policy is set.

In [ ]:
duckdb.sql("""
with headline as (
    select
        count(*)                              as total_trips,
        count(distinct vendorid)              as vendors,
        min(tpep_pickup_datetime)::date       as first_pickup,
        max(tpep_pickup_datetime)::date       as last_pickup,
        sum(total_amount)                     as gross_revenue,
        avg(total_amount)                     as avg_total,
        avg(fare_amount)                      as avg_fare,
        avg(trip_distance)                    as avg_distance,
        avg(tip_amount)                       as avg_tip,
        avg(passenger_count)                  as avg_passengers
    from trips
),
final as (
    select
        total_trips,
        vendors,
        first_pickup,
        last_pickup,
        round(gross_revenue, 0) as gross_revenue,
        round(avg_total, 2)     as avg_total,
        round(avg_fare, 2)      as avg_fare,
        round(avg_distance, 2)  as avg_distance_mi,
        round(avg_tip, 2)       as avg_tip,
        round(avg_passengers, 2) as avg_passengers
    from headline
)
select * from final
""").pl()

## 4. Column Profile — ranges, averages, nulls in one pass

`summarize` gives min / max / avg / approx-unique / null% per column at once —
exactly the "ranges + averages + dirtiness" view we want. Read the extremes
column by column: that's where the data-quality work announces itself.

In [ ]:
# summarize is a built-in profiling command; one scan covers ranges, avgs and null%.
duckdb.sql("summarize trips").pl()

## 5. How Dirty Is It? — explicit validity flags

`summarize` shows *distributions*; this quantifies *rule violations* against the
TLC data dictionary so we can size each cleaning decision before making it.

In [ ]:
duckdb.sql("""
with flagged as (
    select
        count(*)                                                             as total_rows,
        count(*) filter (where fare_amount < 0)                              as negative_fare,
        count(*) filter (where total_amount < 0)                             as negative_total,
        count(*) filter (where trip_distance <= 0)                           as nonpositive_distance,
        count(*) filter (where trip_distance > 1000)                         as absurd_distance,
        count(*) filter (where passenger_count = 0)                          as zero_passengers,
        count(*) filter (where ratecodeid = 99)                              as ratecode_unknown,
        count(*) filter (where payment_type = 0)                             as payment_type_zero,
        count(*) filter (where tpep_dropoff_datetime <= tpep_pickup_datetime) as bad_duration
    from trips
),
final as (
    select
        total_rows,
        negative_fare,
        negative_total,
        nonpositive_distance, round(100.0 * nonpositive_distance / total_rows, 2) as nonpositive_distance_pct,
        absurd_distance,
        zero_passengers,      round(100.0 * zero_passengers      / total_rows, 2) as zero_passengers_pct,
        ratecode_unknown,
        payment_type_zero,    round(100.0 * payment_type_zero    / total_rows, 2) as payment_type_zero_pct,
        bad_duration
    from flagged
)
select * from final
""").pl()

## 6. This Is a Multi-Year Sample, Not a Month

The decisive structural fact: the file holds ~60k trips per year for 2018–2022
plus a partial January 2023. That makes **year-over-year** the natural axis and
puts the **2020 demand shock** at the center of the story.

In [ ]:
duckdb.sql("""
with yearly as (
    select
        year(tpep_pickup_datetime) as pickup_year,
        count(*)                   as trips
    from trips
    group by pickup_year
),
final as (
    select
        pickup_year,
        trips,
        round(100.0 * trips / sum(trips) over (), 2) as pct_of_sample
    from yearly
    order by pickup_year
)
select * from final
""").pl()